In [ ]:
from google.colab import drive
import os,sys
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
sys.path.append('/content/drive/MyDrive/Github_proj/Deep-Learning-Study/ViT')

In [ ]:
%%writefile /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

import math
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter,defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import Dataset,Subset,DataLoader,random_split
from tqdm.notebook import tqdm
from torch.optim.lr_scheduler import ReduceLROnPlateau
import itertools
import regex as re
import pickle

In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

__all__=[
  'load_model',
  'train_model',
  'make_loss_graph'
]

In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def load_model():
  pass

In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def calc_loss_batch(input_batch,target_batch,model,criterion,device):
  #label_smoothing ??
  input_batch,target_batch = input_batch.to(device),target_batch.to(device)
  logits=model(input_batch)
  loss=criterion(logits,target_batch)
  return loss.item()

In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def calc_loss_loader(data_loader,model,criterion,device,num_batches=None):
  total_loss=0.0
  if len(data_loader) == 0:
    return float("nan")
  elif num_batches is None:
    num_batches=len(data_loader)
  else:
    num_batches=min(num_batches,len(data_loader))

  for i,(src,trg) in enumerate(data_loader):
    if i >= num_batches:
      break

    loss=calc_loss_batch(src,trg,model,device)
    total_loss+=loss.item()

  return total_loss/num_batches


In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def evaluate(model,data_loader,criterion,device):
  model.eval()
  with torch.no_grad():
    loss=calc_loss_loader(data_loader,model,criterion,device)
  model.train()
  return loss


In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def train_model(
    models,
    optimizers,
    schedulers,
    criterion,
    train_loader,
    val_loader,
    epochs,
    resume_cfg,
    train_loss_lists=None,
    val_loss_lists=None,
    device
):

  model_names = list(models.keys())
  steps_per_epoch = len(train_loader)

  resume_epoch = resume_cfg['epoch']
  resume_model_idx = resume_cfg['model_idx']
  batch_offset = resume_cfg['batch_offset']
  running_loss = resume_cfg['running_loss']

  for epoch in tqdm(range(resume_epoch,epochs), desc="Epoch", initial=resume_epoch,total=epochs):
    for idx,name in enumerate(model_names):

      if epoch==resume_epoch and idx<resume_model_idx:
        continue

      model = models[name]
      optimizer = optimizers[name]
      scheduler = schedulers[name]

      start=batch_offset if (epoch==resume_epoch and idx==resume_model_idx) else 0

      # 모든 모델이 같은 셔플 순서를 보도록 에폭 기준으로 시드 고정
      #g.manual_seed(base_seed + epoch)
      it=itertools.islice(train_loader,start,None)

      train_loss= running_loss if start > 0 else 0.0
      model.train()

      for step,(train_batch,target_batch) in enumerate(tqdm(it,desc=f"Train {name}", leave=False, initial=start, total=steps_per_epoch),start=start):
        optimizer.zero_grad()
        train_batch,target_batch = train_batch.to(device),target_batch.to(device)

        loss=calc_loss_batch(train_batch,target_batch,model,criterion,device)
        loss.backward()
        optimizer.step()
        scheduler.step()

        # val_loss하고 train_loss 처리 해야 됨.



In [ ]:
%%writefile -a /content/drive/MyDrive/Github_proj//Deep-Learning-Study/ViT/train.py

def make_loss_graph():
  pass